# Poda estructurada con reajuste en Colab

Poda el baseline de SH17 y lo reajusta (apartado 5.3.2.2). Sin reajuste, la poda al 67,5 % deja el
mAP prácticamente en cero.

Se ejecuta de arriba abajo en una sesión limpia. Solo hay que editar la celda de la sección 11, donde
se elige la variante.

| # | Celda | Qué hace |
|---|---|---|
| 1 | Drive | Monta Drive: dataset, pesos del baseline y salida de la campaña |
| 2 | Repositorio | Clona el repositorio |
| 3 | Dependencias | Instala y comprueba los paquetes |
| 4 | Dataset | Descomprime la copia redimensionada |
| 5 | Referencia | Trae el COCO y las particiones y comprueba que son las del estudio |
| 6 | Comprobación | `scripts/00_setup_check.py` |
| 7 | Sesión | Registra la GPU y los núcleos asignados |
| 8 | Pesos | Trae el baseline de Drive y lo comprueba |
| 9 | Estructura | Poda sin entrenar y compara con lo medido en local |
| 10 | Presupuesto | Calcula el coste de la campaña con el coste medido del baseline |
| 11 | Campaña | Poda y reajuste, una variante por ejecución |
| 12 | Recogida | Copia registro, JSON y checkpoints a Drive |

## Preparación

En `MyDrive/tfm-edge/` tienen que estar:

1. `sh17_resized640.tar.gz`, el dataset redimensionado.
2. `runs/s_baseline_default/weights/best.pt`, los pesos del baseline. La importancia de cada canal
   se calcula sobre estos pesos.
3. `sh17_reference.tar.gz`, el COCO de referencia y las particiones (1,8 MiB). Se genera en local,
   donde está `data/raw`: `tar -C data -czf sh17_reference.tar.gz processed/sh17_coco.json splits`.

## Reanudación

Si la sesión se corta durante un reajuste, esa variante se pierde y hay que repetirla. El entrenador
conserva el modelo en memoria para que la poda no se pierda, y por eso la reanudación de Ultralytics
no se aplica. La campaña va una variante por ejecución, por orden de prioridad, para que un corte
solo afecte a la variante en curso.

Relanzar una variante interrumpida falla con el motivo; para rehacerla hay que añadir `--no-resume`.

## 1. Montar Drive

Drive guarda el dataset y los checkpoints, que tienen que sobrevivir al cierre de la sesión, así que
la campaña escribe directamente ahí.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/tfm-edge")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

DATASET_TAR = DRIVE_BASE / "sh17_resized640.tar.gz"
RUNS_DIR = DRIVE_BASE / "runs"
RUNS_DIR.mkdir(exist_ok=True)

assert DRIVE_BASE.is_dir(), f"No se ha montado Drive en {DRIVE_BASE}"
print(f"Drive montado. Base del proyecto: {DRIVE_BASE}")
print(f"  dataset esperado : {DATASET_TAR}  ({'presente' if DATASET_TAR.is_file() else 'FALTA'})")
print(f"  checkpoints en   : {RUNS_DIR}")

if not DATASET_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{DATASET_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n"
        "Súbelo antes de continuar (ver la portada de este cuaderno)."
    )

Salida esperada: la ruta de Drive y el dataset como `presente`. Si aparece `FALTA`, la celda se
detiene y muestra el contenido del directorio.

## 2. Clonar el repositorio

El repositorio es público y se clona sin credenciales. Si existe el secreto `GITHUB_TOKEN`, se usa
en la URL de clonado y los mensajes de error de git se censuran antes de mostrarlos.

In [ ]:
import shutil
import subprocess

from google.colab import userdata

REPO_DIR = Path("/content/tfm-edge-object-detection")

# El repositorio es público, así que el secreto 'GITHUB_TOKEN' es opcional.
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    GITHUB_TOKEN = ""


def _censurar(texto: str) -> str:
    """Quita el token de un texto antes de mostrarlo."""
    return texto.replace(GITHUB_TOKEN, "[TOKEN]") if GITHUB_TOKEN else texto


if REPO_DIR.is_dir():
    print(f"El repositorio ya está en {REPO_DIR}; se actualiza.")
    resultado = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
else:
    resultado = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            f"https://{GITHUB_TOKEN + '@' if GITHUB_TOKEN else ''}github.com/ajgomez23/tfm-compresion-yolov8.git",
            str(REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

if resultado.returncode != 0:
    raise RuntimeError(
        "Git ha fallado (mensaje censurado por si contenía el token):\n"
        + _censurar(resultado.stderr)[:800]
    )

# Se comprueba que el clon trae lo que hace falta, no solo que git dijera que si.
imprescindibles = [
    "pyproject.toml",
    "requirements.txt",
    "scripts/00_setup_check.py",
    "scripts/04_train_baselines.py",
    "src/tfm_edge",
]
faltan = [f for f in imprescindibles if not (REPO_DIR / f).exists()]
if faltan:
    raise RuntimeError(f"El clon está incompleto: faltan {faltan}")

revision = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repositorio listo en {REPO_DIR}")
print(f"  revisión: {revision}")

Salida esperada: la revisión (hash y asunto del último commit), que identifica la versión del código
usada.

## 3. Instalar las dependencias

`requirements.txt` instala las versiones fijadas (Ultralytics incluido) y `pip install .` instala el
paquete `tfm_edge`. Se usa la instalación normal y no la editable porque en Colab `pip install -e .`
terminaba sin error pero `import tfm_edge` fallaba.

El `%cd` es necesario: `pip install .` instala el directorio actual.

In [ ]:
%cd /content/tfm-edge-object-detection
!pip install -q -r requirements.txt
!pip install -q .

In [ ]:
import importlib
import subprocess
import sys
import tempfile

# Se reinician los módulos ya cargados: pip puede haber sustituido versiones que
# esta sesión de Python tenía importadas de antes.
for modulo in [m for m in list(sys.modules) if m.startswith(("tfm_edge", "ultralytics"))]:
    del sys.modules[modulo]

fallos = []
for paquete, esperada in [("torch", None), ("ultralytics", "8.3.253")]:
    try:
        modulo = importlib.import_module(paquete)
        version = getattr(modulo, "__version__", "sin versión")
        marca = "OK" if not esperada or version == esperada else f"AVISO: se esperaba {esperada}"
        print(f"  {paquete:12s} {version:12s} {marca}")
    except Exception as error:
        fallos.append(f"{paquete}: {type(error).__name__}: {error}")

# 'import tfm_edge' se comprueba en un subproceso y desde un directorio neutro, porque
# el %cd ya ha añadido el repositorio a sys.path en este kernel.
with tempfile.TemporaryDirectory() as neutral:
    prueba = subprocess.run(
        [sys.executable, "-c", "import tfm_edge; print(tfm_edge.__file__)"],
        capture_output=True,
        text=True,
        cwd=neutral,
    )
if prueba.returncode == 0:
    print(f"  {'tfm_edge':12s} importable desde un intérprete limpio")
    print(f"               {prueba.stdout.strip()}")
else:
    fallos.append(
        "tfm_edge no es importable desde un intérprete limpio:\n      "
        + (prueba.stderr.strip().splitlines()[-1] if prueba.stderr else "sin detalle")
        + "\n      Reinstala con: !pip install --force-reinstall --no-deps ."
    )

import torch

print(f"  CUDA disponible: {torch.cuda.is_available()}")

if fallos:
    raise RuntimeError("Problemas de instalación:\n  - " + "\n  - ".join(fallos))
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU en esta sesión. Cámbialo en Entorno de ejecución, Cambiar tipo de "
        "entorno de ejecución, GPU."
    )
print("\nDependencias correctas.")

Salida esperada: las versiones de torch y ultralytics, `tfm_edge` importable desde un intérprete
limpio y `CUDA disponible: True`.

In [ ]:
# requirements.txt fija numpy por debajo de 2.0 (lo exige el extra "export" de
# Ultralytics) y Colab trae la serie 2.x. pip baja la versión, pero los paquetes ya
# instalados siguen compilados contra la ABI anterior y pueden fallar después de
# importarse. Cada paquete se ejercita en un subproceso propio.
from tfm_edge.common.binary_compat import (
    check_numpy_binary_compat,
    format_binary_compat_report,
)

resultados = check_numpy_binary_compat()
print(format_binary_compat_report(resultados))

rotos = [r for r in resultados if r.installed and not r.healthy]
if rotos:
    raise RuntimeError(
        "Hay paquetes incompatibles con la NumPy instalada. Aplica el remedio de "
        "arriba y reinicia el entorno de ejecución antes de volver a ejecutar, porque "
        "el NumPy anterior sigue cargado en memoria."
    )

Salida esperada: las cinco líneas de la tabla con `[ OK ]`.

`[ ABI]` indica que el paquete está compilado contra otra versión de NumPy y la celda se detiene.
Hay que reiniciar el entorno de ejecución después de reinstalar. `[ -- ]` indica un paquete ausente,
que ya informa la celda anterior.

## 4. Descomprimir el dataset

El `.tar.gz` se descomprime en el disco local de la sesión. Leer miles de imágenes desde Drive haría
de la carga de datos el cuello de botella.

In [ ]:
import tarfile
import time

DATA_DIR = REPO_DIR / "data"
DATASET_DIR = DATA_DIR / "resized_640"
DATA_DIR.mkdir(exist_ok=True)

if DATASET_DIR.is_dir() and (DATASET_DIR / "sh17_ultralytics.yaml").is_file():
    print(f"El dataset ya está en {DATASET_DIR}; no se vuelve a descomprimir.")
else:
    inicio = time.perf_counter()
    print(f"Descomprimiendo {DATASET_TAR.name} ({DATASET_TAR.stat().st_size / 2**30:.2f} GiB)...")
    with tarfile.open(DATASET_TAR, "r:gz") as tar:
        tar.extractall(DATA_DIR)
    print(f"  hecho en {time.perf_counter() - inicio:.0f} s")

# Se comprueba el contenido, no solo que exista el directorio.
yaml_dataset = DATASET_DIR / "sh17_ultralytics.yaml"
num_imagenes = len(list((DATASET_DIR / "images").glob("*")))
num_etiquetas = len(list((DATASET_DIR / "labels").glob("*.txt")))

problemas = []
if not yaml_dataset.is_file():
    problemas.append(f"falta {yaml_dataset}")
if num_imagenes != 8099:
    problemas.append(f"hay {num_imagenes} imágenes y se esperaban 8.099")
if num_etiquetas != 8099:
    problemas.append(f"hay {num_etiquetas} etiquetas y se esperaban 8.099")
if problemas:
    raise RuntimeError("El dataset descomprimido no es correcto: " + "; ".join(problemas))

print(f"Dataset verificado: {num_imagenes:,} imágenes, {num_etiquetas:,} etiquetas")
print(f"  YAML: {yaml_dataset}")

In [ ]:
# Un tar.gz generado con una versión antigua puede llevar rutas absolutas.
from tfm_edge.data.prepare import assert_dataset_portable

try:
    assert_dataset_portable(DATASET_DIR)
    print("Portabilidad verificada: ninguna ruta absoluta en el YAML ni en las listas")
except Exception as error:
    raise RuntimeError(
        f"{error}\n\n"
        "El tar.gz de Drive se generó con rutas absolutas. "
        "Regenéralo en local y súbelo de nuevo:\n"
        "    python scripts/02b_resize_dataset.py --target-size 640 --force\n"
        "    tar -C data -czf sh17_resized640.tar.gz resized_640"
    ) from None

In [ ]:
# Ultralytics tiene que resolver las rutas contra esta ubicación.
from ultralytics.data.utils import check_det_dataset

datos = check_det_dataset(str(yaml_dataset), autodownload=False)
print(f"Ultralytics acepta el dataset: {datos['nc']} clases")
print(f"  train: {datos['train']}")
print(f"  val  : {datos['val']}")
assert datos["nc"] == 17, f"Se esperaban 17 clases y hay {datos['nc']}"
assert Path(datos["val"]).is_file(), "La lista de validación no se resuelve"
assert str(DATASET_DIR) in str(datos["val"]), (
    f"Las rutas apuntan fuera de esta sesión: {datos['val']}. " "El artefacto no es portable."
)
print("\nLas rutas se resuelven contra esta sesión, no contra la máquina de origen.")

Salida esperada: 8.099 imágenes, 17 clases y las rutas de `train` y `val` dentro de
`/content/.../data/resized_640/`. Si apuntan a otra máquina, hay que regenerar el `tar.gz`; la celda
anterior lo detecta e indica el comando.

## 5. La referencia de evaluación

Las métricas las calcula pycocotools sobre `data/processed/sh17_coco.json`, con la partición de
`data/splits/val.txt`. Ninguno de los dos está en el repositorio (`data/` no se versiona) ni en
`sh17_resized640.tar.gz`, así que sin ellos la campaña podaría y fallaría al evaluar.

No se pueden regenerar aquí: `scripts/02_prepare_dataset.py` valida la procedencia y rechaza una
copia redimensionada (exige resoluciones variadas y un ancho máximo de al menos 1920 px). Además, las
áreas del COCO están en coordenadas originales, de las que depende el criterio de tamaño de objeto.
Por eso el COCO se copia desde Drive (1,8 MiB comprimido).

In [ ]:
import hashlib
import json
import tarfile

# Huellas de los ficheros con los que se evaluó el baseline en local, para medir contra
# la misma referencia que el resto del capítulo 5.
REFERENCIA_ESPERADA = {
    "processed/sh17_coco.json": "f67e72991eac18e1b6573a7c790ccb4b1a29b6c2fdc34d1f2d34526818cc99e7",
    "splits/val.txt": "cb1d27579cb1d218b02c7d64125ef149316cd6ae7234243d01c0306a472ff743",
    "splits/train.txt": "0adc1a8a30bf8f0d18803e98d1e3056b45f7c918d872632e2e87e92ddf10db02",
    "splits/test.txt": "a75a91888f71ab588c384929d7ee1e3fd22cc262832be370d0fcc089ab2e25bc",
}

REFERENCE_TAR = DRIVE_BASE / "sh17_reference.tar.gz"

if not REFERENCE_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{REFERENCE_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n\n"
        "Se genera en local, donde está data/raw, y se sube a "
        "Drive una sola vez:\n"
        "    tar -C data -czf sh17_reference.tar.gz processed/sh17_coco.json splits\n"
        "Son 1,8 MiB. No se puede regenerar en Colab: ver el texto de esta sección."
    )

with tarfile.open(REFERENCE_TAR, "r:gz") as tar:
    tar.extractall(DATA_DIR)
print(f"Referencia extraída de {REFERENCE_TAR.name} en {DATA_DIR}")

# Se comprueba cada fichero y no el tar.gz, cuyo hash depende de gzip y de las marcas de
# tiempo.
print()
discrepancias = []
for relativa, esperado in REFERENCIA_ESPERADA.items():
    ruta = DATA_DIR / relativa
    if not ruta.is_file():
        discrepancias.append(f"{relativa}: no se ha extraído")
        continue
    obtenido = hashlib.sha256(ruta.read_bytes()).hexdigest()
    marca = "OK  " if obtenido == esperado else "MAL "
    print(f"  {marca} {relativa:34s} {obtenido[:16]}...")
    if obtenido != esperado:
        discrepancias.append(f"{relativa}: {obtenido[:16]}... != {esperado[:16]}...")

if discrepancias:
    raise RuntimeError(
        "La referencia de Drive no es la del estudio:\n  "
        + "\n  ".join(discrepancias)
        + "\n\nVuelve a generar el tar.gz en local desde el mismo data/processed y "
        "data/splits con los que se evaluó el baseline."
    )

# Recuentos esperados de la referencia y de la partición de validación.
COCO_PATH = DATA_DIR / "processed" / "sh17_coco.json"
SPLITS_DIR = DATA_DIR / "splits"

coco = json.loads(COCO_PATH.read_text(encoding="utf-8"))
num_val = len([x for x in (SPLITS_DIR / "val.txt").read_text().splitlines() if x.strip()])
print()
print(
    f"COCO   : {len(coco['images']):,} imágenes, {len(coco['annotations']):,} anotaciones, "
    f"{len(coco['categories'])} clases"
)
print(f"val.txt: {num_val:,} imágenes")
assert (len(coco["images"]), len(coco["annotations"]), len(coco["categories"])) == (8099, 75994, 17)
assert num_val == 1213, f"val.txt trae {num_val} imágenes y se esperaban 1.213"

# La misma comprobación que hace el script antes de podar.
from tfm_edge.evaluation.runner import assert_evaluation_inputs_available

comprobadas = assert_evaluation_inputs_available(COCO_PATH, SPLITS_DIR, "val", yaml_dataset)
print("\nLas tres piezas de la evaluación están:")
for nombre, ruta in comprobadas.items():
    print(f"  {nombre}: {ruta}")

Salida esperada: las cuatro líneas `OK`, las 8.099 imágenes con 75.994 anotaciones y las 1.213 de
validación. Si alguna huella sale `MAL`, la referencia es distinta de la del baseline y no hay que
continuar.

## 6. Comprobación del entorno

`scripts/00_setup_check.py` comprueba versiones, dependencias, rutas y GPU. Su salida queda en el
cuaderno como registro del entorno de la sesión.

In [ ]:
!python scripts/00_setup_check.py

Salida esperada: el informe completo del entorno en el que se obtienen los resultados.

## 7. Máquina asignada

Colab asigna máquinas distintas en cada sesión y eso cambia los tiempos. La huella se guarda junto a
los checkpoints, en Drive.

In [ ]:
import json
import os
import subprocess

import torch


def _cores_fisicos() -> int:
    """Cuenta núcleos físicos leyendo /proc/cpuinfo."""
    try:
        texto = Path("/proc/cpuinfo").read_text()
        ids = {
            linea.split(":")[1].strip()
            for linea in texto.splitlines()
            if linea.startswith("core id")
        }
        return len(ids) or os.cpu_count() or 0
    except Exception:
        return os.cpu_count() or 0


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()

huella = {
    "gpu_nvidia_smi": gpu,
    "gpu_torch": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "gpu_memory_gib": (
        round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
        if torch.cuda.is_available()
        else ""
    ),
    "cpu_logicos": os.cpu_count(),
    "cpu_fisicos": _cores_fisicos(),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
}

for clave, valor in huella.items():
    print(f"  {clave:16s} {valor}")

destino_huella = RUNS_DIR / "session_environment.json"
historico = []
if destino_huella.is_file():
    historico = json.loads(destino_huella.read_text())
historico.append(huella)
destino_huella.write_text(json.dumps(historico, indent=2, ensure_ascii=False))
print(f"\nHuella añadida a {destino_huella} (sesión {len(historico)})")

Salida esperada: GPU, núcleos físicos y RAM. El baseline se entrenó en una Tesla T4 con 4 núcleos
físicos a 116,5 s por época de extremo a extremo; con otra máquina cambia el presupuesto de la
sección 10.

## 8. Pesos de partida

Antes de lanzar la campaña se comprueba el checkpoint: número de parámetros, número de clases y que
los pesos están en FP32. Un checkpoint en FP16 se podaría sin error, pero los resultados no serían
comparables con el resto del estudio.

In [ ]:
import shutil

from tfm_edge.models.detector import Detector
from tfm_edge.training.trainer import verify_checkpoint_is_fp32

# Valores medidos en local sobre este mismo fichero.
BASELINE_PARAMS = 11_142_179
BASELINE_CLASSES = 17
BASELINE_EXPERIMENT_ID = "s_baseline_default_745e3d08"

origen_pesos = RUNS_DIR / "s_baseline_default"
destino_pesos = REPO_DIR / "models" / "baselines" / "s_baseline_default"
destino_pesos.mkdir(parents=True, exist_ok=True)

CHECKPOINT = destino_pesos / "best.pt"
if not CHECKPOINT.is_file():
    fuente = origen_pesos / "weights" / "best.pt"
    if not fuente.is_file():
        raise FileNotFoundError(
            f"Falta '{fuente}'.\n"
            f"Lo que hay en {origen_pesos}: "
            f"{sorted(p.name for p in origen_pesos.glob('*')) if origen_pesos.is_dir() else 'nada'}"
        )
    shutil.copy2(fuente, CHECKPOINT)
    print(f"Copiado {fuente} en {CHECKPOINT}")

# Ficheros que acompañan al checkpoint: el coste de entrenamiento (para el
# presupuesto) y la configuración con la que se entrenó.
for nombre in ("training_cost.json", "args.yaml", "complexity.json", "schedule.json"):
    fuente = origen_pesos / nombre
    if fuente.is_file() and not (destino_pesos / nombre).is_file():
        shutil.copy2(fuente, destino_pesos / nombre)

print(f"\nCheckpoint: {CHECKPOINT}  ({CHECKPOINT.stat().st_size / 2**20:.1f} MiB)")
print(f"Acompañado de: {sorted(p.name for p in destino_pesos.glob('*') if p.name != 'best.pt')}")

informe_fp32 = verify_checkpoint_is_fp32(CHECKPOINT)
print(f"\nPrecisión verificada: {informe_fp32}")

detector = Detector.load(CHECKPOINT, input_resolution=640)
parametros = detector.count_parameters()
clases = detector.num_classes

print(f"Parámetros : {parametros:,}   (se esperaban {BASELINE_PARAMS:,})")
print(f"Clases     : {clases}   (se esperaban {BASELINE_CLASSES})")

problemas = []
if parametros != BASELINE_PARAMS:
    problemas.append(f"parámetros {parametros:,} != {BASELINE_PARAMS:,}")
if clases != BASELINE_CLASSES:
    problemas.append(f"clases {clases} != {BASELINE_CLASSES}")
if problemas:
    raise RuntimeError(
        "El checkpoint de Drive no es el baseline del estudio: " + "; ".join(problemas)
    )
print("\nEs el baseline del estudio.")

Salida esperada: parámetros, clases y precisión verificada, coincidentes con lo esperado. Si no, la
celda se detiene.

## 9. Comprobación de la estructura podada

Poda sin entrenar y sin escribir en el registro, y compara el número de parámetros con el medido en
local. Comprueba que las versiones de `torch-pruning` y Ultralytics instaladas reparten el recorte
igual, de modo que el modelo que se reajusta aquí es el mismo que se midió allí. Si no coinciden, no
hay que continuar.

In [ ]:
import subprocess
import sys

import torch

# Parámetros medidos en local (torch 2.8.0, torch-pruning 1.6.0, ultralytics 8.3.253)
# con semilla por rama, que hace reproducible también el control aleatorio.
#
# Se miden sin reajuste. En las variantes de una etapa la arquitectura solo depende
# de los pesos de partida. La iterativa reajusta entre ciclos y la importancia L1 se
# calcula sobre los pesos, así que con reajuste el resultado final cambia (3.711.019
# frente a 3.503.352); aquí solo sirve para comprobar que esta máquina poda igual.
ESTRUCTURA_ESPERADA = {
    "l1_global_r25.yaml": {"l1": 6_781_449, "random": 6_839_035},
    "l1_global_r50.yaml": {"l1": 3_618_872, "random": 3_626_541},
    "bn_scale_r50.yaml": {"bn_gamma": 3_612_597, "random": 3_626_541},
    # Sin reajuste parcial. La campaña real con --finetune da 3.711.019.
    "iterative_r50_4steps.yaml": {"l1": 3_503_352, "random": 3_580_898},
}

CONFIG_A_COMPROBAR = "l1_global_r50.yaml"  # la variante de prioridad 1

comprobacion = Path("/content/comprobacion_estructura")
comando = [
    sys.executable,
    "scripts/13_prune.py",
    "--checkpoint",
    str(CHECKPOINT),
    "--config",
    f"configs/compression/pruning/{CONFIG_A_COMPROBAR}",
    "--random-control",  # también se comprueba el control aleatorio
    "--no-evaluate",  # no mide mAP: aquí solo interesa la estructura
    "--dry-run",  # no toca el registro
    "--output-dir",
    str(comprobacion / "models"),
    "--raw-dir",
    str(comprobacion / "raw"),
]
print(" ".join(comando), "\n")
codigo = subprocess.run(comando, cwd=REPO_DIR).returncode
if codigo != 0:
    raise RuntimeError(f"La poda de comprobación ha terminado con código {codigo}.")

# --dry-run no escribe el JSON de detalle, pero sí el checkpoint podado, así que los
# parámetros se cuentan sobre el checkpoint.
checkpoints = sorted((comprobacion / "models").glob("*/finetune/pruned_before_finetune.pt"))
if not checkpoints:
    raise RuntimeError(
        f"No hay ningún checkpoint podado bajo {comprobacion / 'models'}. "
        "La poda no ha llegado a producir nada; revisa la salida de arriba."
    )

esperados = ESTRUCTURA_ESPERADA[CONFIG_A_COMPROBAR]
(criterio_configurado,) = [c for c in esperados if c != "random"]

obtenidos = {}
for ruta in checkpoints:
    experimento = ruta.parent.parent.name
    criterio = "random" if "_randomctl_" in experimento else criterio_configurado
    contenido = torch.load(ruta, map_location="cpu", weights_only=False)
    modelo = contenido["model"] if isinstance(contenido, dict) else contenido
    obtenidos[criterio] = sum(p.numel() for p in modelo.parameters())

print()
print(f"{'criterio':<12} {'aquí':>14} {'estación':>14}")
print("-" * 42)
discrepancias = []
for criterio, esperado in esperados.items():
    obtenido = obtenidos.get(criterio)
    texto = "ausente" if obtenido is None else format(obtenido, ",d")
    marca = "" if obtenido == esperado else "   (no coincide)"
    print(f"{criterio:<12} {texto:>14} {esperado:>14,d}{marca}")
    if obtenido != esperado:
        discrepancias.append((criterio, texto, esperado))

if discrepancias:
    detalle = "; ".join(f"{c}: {t} frente a {e:,d}" for c, t, e in discrepancias)
    raise RuntimeError(
        f"La poda no reparte igual aquí que en local ({detalle}).\n"
        "Un reajuste sobre otra arquitectura no es comparable con lo ya medido. Si "
        "solo difiere el control aleatorio, comprueba que se usa la semilla por "
        "rama; si difiere el criterio configurado, compara las versiones de torch, "
        "torch-pruning y ultralytics con las de la celda 3."
    )
print("\nMisma arquitectura que la medida en local, criterio y control.")

Salida esperada: las cifras de parámetros y la reducción medida. Al 50 % pedido salen 3.618.872
parámetros, un 67,5 % de reducción real, porque podar un canal arrastra las entradas que dependen de
él.

## 10. Plan y presupuesto

Orden de `PLAN`:

1. `l1_global_r50.yaml` con control aleatorio. Sin reajuste, las variantes al 67,5 % quedan en
   valores del orden de 1e-7, que no permiten comparar criterios.
2. `l1_global_r25.yaml` con control aleatorio. Al 39,1 % la comparación con el azar ya es clara sin
   reajuste (L1 conserva el 26 % del mAP@50-95 y el azar nada); falta medir cuánto se recupera.
3. `iterative_r50_4steps.yaml`, que pone a prueba la hipótesis del 9.3.4 con reajuste parcial entre
   ciclos.
4. `bn_scale_r50.yaml`, tercer criterio al mismo ratio.

El presupuesto se calcula con el coste medido del baseline (`training_cost.json`). Una variante con
control aleatorio son dos reajustes; el control es necesario para atribuir el resultado al criterio
(9.3.1).

In [ ]:
# El plan de la campaña, en orden de prioridad: (configuración, con control).
PLAN = [
    ("l1_global_r50.yaml", True),
    ("l1_global_r25.yaml", True),
    ("iterative_r50_4steps.yaml", False),
    ("bn_scale_r50.yaml", False),
]

In [ ]:
from tfm_edge.common.config import load_config
from tfm_edge.training.cost import read_training_cost, seconds_per_epoch

coste_baseline = read_training_cost(CHECKPOINT)
segundos_epoca = seconds_per_epoch(coste_baseline)

if segundos_epoca is None:
    print(
        "AVISO: no hay coste de entrenamiento junto al checkpoint, así que no se "
        "puede presupuestar la campaña con una cifra medida. Se lanza igual; el "
        "coste real lo dejará cada campaña en su propio training_cost.json."
    )
else:
    print(
        f"Baseline: {coste_baseline['training_epochs']} épocas en "
        f"{coste_baseline['training_time_s']:,.0f} s sobre {coste_baseline['training_device']}"
    )
    print(f"  {segundos_epoca:.1f} s por época\n")

    print(f"{'configuración':<30} {'épocas':>7} {'variantes':>10} {'cota superior':>15}")
    print("-" * 66)
    total = 0.0
    for nombre, con_control in PLAN:
        config = load_config(REPO_DIR / "configs" / "compression" / "pruning" / nombre)
        poda = config.pruning
        # La iterativa reajusta entre ciclos, además del reajuste final.
        epocas = poda.finetune_epochs + poda.finetune_epochs_per_step * (poda.iterative_steps - 1)
        variantes = 2 if con_control else 1
        segundos = epocas * segundos_epoca * variantes
        total += segundos
        print(f"{nombre:<30} {epocas:>7} {variantes:>10} {segundos / 3600:>14.2f} h")
    print("-" * 66)
    print(f"{'TOTAL':<30} {'':>7} {'':>10} {total / 3600:>14.2f} h")
    print(
        "\nEs una cota superior: se calcula con el coste por época del modelo "
        "entero, y el podado tiene entre un 39 % y un 69 % menos de parámetros. "
        "El coste real de cada campaña queda en su training_cost.json."
    )

Salida esperada: la tabla completa. El baseline costó 116,5 s por época en la T4 de la sesión
original; con otra máquina la cota cambia.

## 11. Campaña

Una variante por ejecución: cambia `VARIANTE` y ejecuta las dos celdas. Al terminar, ejecuta la
sección 12.

In [ ]:
# Configuración de la campaña.
VARIANTE = 0  # índice dentro de PLAN
LOTE = 16  # el mismo con el que se entrenó el baseline
WORKERS = 2  # medido: 2 núcleos físicos en la máquina de Colab
REHACER = False  # True solo para rehacer una variante ya empezada


CONFIG, CON_CONTROL = PLAN[VARIANTE]
SALIDA_PODA = DRIVE_BASE / "pruned"

print(f"Variante {VARIANTE}: {CONFIG}" + ("  + control aleatorio" if CON_CONTROL else ""))
print(f"Salida (en Drive): {SALIDA_PODA}")

In [ ]:
# Se lanza con subprocess y no con la magia '!' por lo mismo que en el cuaderno
# 00: así se puede comprobar el código de salida.
import shutil
import subprocess
import sys
from pathlib import Path


def salvar_registro():
    """Copia el registro y los volcados crudos a Drive y devuelve una línea por directorio.

    Se llama siempre, también si la campaña falla, porque el disco de la sesión se pierde.
    """
    destino = DRIVE_BASE / "resultados_poda"
    destino.mkdir(parents=True, exist_ok=True)
    lineas = []
    for subdir in ("registry", "raw/pruning"):
        origen = REPO_DIR / "results" / subdir
        if origen.is_dir():
            copia = destino / Path(subdir).name
            shutil.copytree(origen, copia, dirs_exist_ok=True)
            lineas.append(f"{origen} copiado a {copia}: {sorted(p.name for p in copia.glob('*'))}")
        else:
            lineas.append(f"AVISO: no existe {origen}")
    return lineas


comando = [
    sys.executable,
    "scripts/13_prune.py",
    "--checkpoint",
    str(CHECKPOINT),
    "--config",
    f"configs/compression/pruning/{CONFIG}",
    "--parent-experiment-id",
    BASELINE_EXPERIMENT_ID,
    "--finetune",
    "--batch-size",
    str(LOTE),
    "--workers",
    str(WORKERS),
    # La salida va a Drive para conservar las variantes terminadas si la sesión se corta.
    "--output-dir",
    str(SALIDA_PODA),
    # El mAP se mide en la GPU de esta sesión y el device_id lo indica. Los checkpoints
    # se vuelven a evaluar después en local.
    "--device-id",
    "colab_t4_gpu",
]
if CON_CONTROL:
    comando.append("--random-control")
if REHACER:
    comando.append("--no-resume")

print(" ".join(comando), "\n")
codigo = subprocess.run(comando, cwd=REPO_DIR).returncode

# Se copia a Drive antes de comprobar el resultado, también si ha fallado.
print("-" * 78)
for linea in salvar_registro():
    print(linea)
print("-" * 78)

if codigo == 0:
    print("\nCampaña completada e inscrita en el registro.")
elif codigo == 2:
    print(
        "\nAVISO: la poda ha fallado de forma controlada y ha quedado inscrita "
        "con status=fallido y el motivo: ese ratio no es aplicable a esta "
        "arquitectura."
    )
else:
    raise RuntimeError(
        f"La campaña ha terminado con código {codigo}. Revisa el error de más "
        "arriba. Si indica que no se puede reanudar, pon REHACER=True para rehacer "
        "la variante desde el principio. Lo inscrito hasta el fallo ya está copiado "
        "a Drive."
    )

Salida esperada:

- Las métricas antes y después del reajuste: la primera mide cuánto degrada la poda y la segunda
  cuánto se recupera.
- La reducción medida de parámetros.
- Las líneas `Checkpoint verificado en FP32` y la comprobación de que la poda se conserva al guardar.
- Con control aleatorio, sus métricas junto a las del criterio configurado.

## 12. Recogida de resultados

El registro y los volcados crudos ya se copian al terminar cada campaña. Esta celda repite la copia
y lista lo que hay:

- El registro, del que salen las tablas.
- Los JSON crudos de cada campaña, con el detalle por capa y el informe de reajuste.
- Los checkpoints reajustados. La fila de calidad de esta sesión se mide en esta GPU
  (`device_id=colab_t4_gpu`); los checkpoints se vuelven a evaluar en local por el mismo camino que
  el baseline.

In [ ]:
# La copia ya la hizo cada campaña; se repite para listar lo que hay.
for linea in salvar_registro():
    print(linea)

# Los checkpoints ya están en Drive (--output-dir).
if SALIDA_PODA.is_dir():
    print(f"\nCheckpoints en {SALIDA_PODA}:")
    for pesos in sorted(SALIDA_PODA.rglob("*.pt")):
        print(f"  {pesos.relative_to(SALIDA_PODA)}  ({pesos.stat().st_size / 2**20:.1f} MiB)")
else:
    print(f"\nAVISO: no existe {SALIDA_PODA}")

Salida esperada: la lista de CSV copiados y la de checkpoints, que se descargan para incorporarlos al
registro local. De cada variante hacen falta `pruned_before_finetune.pt` (podado sin reajustar) y el
`best.pt` del reajuste.